## Summary

This workflow demonstrates the complete raster tiling process:

1. **Load Raster Data**: Inspect large raster files and their properties
2. **Define Parameters**: Configure tile size and overlap for optimal processing
3. **Split into Tiles**: Create manageable chunks while maintaining spatial reference
4. **Process Tiles**: Apply operations independently to reduce memory usage
5. **Merge Results**: Combine tiles back into a single raster with overlap handling
6. **Validate**: Verify accuracy by comparing with the original raster

### Key Benefits:
- **Memory Efficiency**: Process one tile at a time instead of entire raster
- **Better Accuracy**: Overlaps help reduce edge effects in processing
- **Parallelization**: Tiles can be processed in parallel for speed improvements
- **Scalability**: Works with arbitrarily large rasters

### Next Steps:
You can integrate this tiling approach into your template matching workflow by:
- Processing template matching on each tile separately
- Merging detection results from all tiles
- Running BIRCH clustering on the merged detections

In [ ]:
# Visualize error distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: MSE heatmap
im = axes[0].imshow(mse_per_pixel, cmap='hot')
axes[0].set_title('Pixel-wise MSE (Error Map)', fontsize=12)
plt.colorbar(im, ax=axes[0], label='MSE')

# Plot 2: MSE histogram
axes[1].hist(mse_per_pixel.flatten(), bins=100, edgecolor='black')
axes[1].set_xlabel('MSE Value')
axes[1].set_ylabel('Number of Pixels')
axes[1].set_title('Distribution of Pixel-wise Errors', fontsize=12)
axes[1].set_yscale('log')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('Output/error_analysis.png', dpi=100, bbox_inches='tight')
plt.show()

print("Error analysis visualization saved to Output/error_analysis.png")

In [ ]:
# Visualize differences
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Show original bands
for band_idx in range(3):
    ax = axes[0, band_idx]
    band_data = original[band_idx].astype(np.uint8)
    ax.imshow(band_data, cmap='gray')
    ax.set_title(f'Original Band {band_idx + 1}')
    ax.axis('off')

# Show merged bands
for band_idx in range(3):
    ax = axes[1, band_idx]
    band_data = np.clip(merged_raster[band_idx], 0, 255).astype(np.uint8)
    ax.imshow(band_data, cmap='gray')
    ax.set_title(f'Merged Band {band_idx + 1}')
    ax.axis('off')

plt.suptitle('Comparison: Original vs Merged Bands', fontsize=14, y=0.98)
plt.tight_layout()
plt.savefig('Output/band_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

print("Band comparison visualization saved to Output/band_comparison.png")

In [ ]:
print("Validating merged raster against original...")
print("=" * 60)

# Load original raster for comparison
with rasterio.open(raster_path) as src:
    original = src.read().astype(np.float32)

# Compare statistics
print("\nStatistical Comparison (per band):")
print(f"{'Band':<6} {'Metric':<20} {'Original':<12} {'Merged':<12} {'Difference':<12}")
print("-" * 62)

for band_idx in range(num_bands):
    original_band = original[band_idx]
    merged_band = merged_raster[band_idx]
    
    orig_mean = np.mean(original_band)
    merged_mean = np.mean(merged_band)
    diff_mean = abs(orig_mean - merged_mean)
    
    print(f"Band {band_idx+1:<2} {'Mean':<20} {orig_mean:<12.2f} {merged_mean:<12.2f} {diff_mean:<12.2f}")
    
    orig_std = np.std(original_band)
    merged_std = np.std(merged_band)
    diff_std = abs(orig_std - merged_std)
    
    print(f"{'':6} {'Std Dev':<20} {orig_std:<12.2f} {merged_std:<12.2f} {diff_std:<12.2f}")

# Calculate overall RMSE
rmse = np.sqrt(np.mean((original - merged_raster) ** 2))
print(f"\nOverall RMSE: {rmse:.4f}")

# Calculate MSE per pixel to find problematic areas
mse_per_pixel = np.mean((original - merged_raster) ** 2, axis=0)

print(f"\nPixel-wise MSE Statistics:")
print(f"  Min MSE: {np.min(mse_per_pixel):.6f}")
print(f"  Max MSE: {np.max(mse_per_pixel):.6f}")
print(f"  Mean MSE: {np.mean(mse_per_pixel):.6f}")
print(f"  Median MSE: {np.median(mse_per_pixel):.6f}")

print("=" * 60)

## 7. Validate Results

Compare the merged raster with the original to verify accuracy and data integrity.

In [ ]:
def merge_tiles_simple(tiles, tile_metadata, raster_width, raster_height, num_bands):
    """
    Merge tiles back into single raster, averaging overlapping regions
    
    Args:
        tiles: List of tile arrays
        tile_metadata: List of tile metadata dicts
        raster_width: Width of original raster
        raster_height: Height of original raster
        num_bands: Number of bands
        
    Returns:
        Merged raster array and weight map
    """
    # Initialize output raster and weight map
    merged = np.zeros((num_bands, raster_height, raster_width), dtype=np.float32)
    weights = np.zeros((raster_height, raster_width), dtype=np.float32)
    
    # Place each tile into the merged raster
    for tile, meta in zip(tiles, tile_metadata):
        row_off = meta['row_off']
        col_off = meta['col_off']
        tile_height = meta['height']
        tile_width = meta['width']
        
        # Add tile data
        merged[:, row_off:row_off+tile_height, col_off:col_off+tile_width] += tile
        weights[row_off:row_off+tile_height, col_off:col_off+tile_width] += 1
    
    # Normalize by weights (handle overlapping regions)
    weights[weights == 0] = 1  # Avoid division by zero
    for band in range(num_bands):
        merged[band] = merged[band] / weights
    
    return merged, weights

print("Merging tiles back together...")

with rasterio.open(raster_path) as src:
    num_bands = src.count
    raster_width = src.width
    raster_height = src.height

merged_raster, weight_map = merge_tiles_simple(
    tiles, tile_metadata, raster_width, raster_height, num_bands
)

print(f"Merged raster shape: {merged_raster.shape}")
print(f"Merged raster data type: {merged_raster.dtype}")

# Visualize weight map (shows overlap regions)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# Plot 1: Weight map
im1 = ax1.imshow(weight_map, cmap='viridis')
ax1.set_title('Tile Weight Map (overlap regions)', fontsize=12)
plt.colorbar(im1, ax=ax1, label='Number of overlapping tiles')

# Plot 2: Merged raster (RGB composite)
rgb_merged = merged_raster[:3].astype(np.uint8).transpose(1, 2, 0)
rgb_merged = np.clip(rgb_merged, 0, 255)
ax2.imshow(rgb_merged)
ax2.set_title('Merged Raster (RGB)', fontsize=12)

plt.tight_layout()
plt.savefig('Output/merge_result.png', dpi=100, bbox_inches='tight')
plt.show()

print("Merge result visualization saved to Output/merge_result.png")

## 6. Merge Tiles Back Together

Combine processed tiles back into a single raster while handling overlaps correctly.

In [ ]:
def calculate_tile_statistics(tile_data, tile_meta):
    """
    Example processing function: calculate statistics for each tile
    
    Args:
        tile_data: numpy array of shape (bands, height, width)
        tile_meta: dict with tile metadata
        
    Returns:
        dict with statistics for this tile
    """
    stats = {
        'tile_position': (tile_meta['row_off'], tile_meta['col_off']),
        'band_count': tile_data.shape[0],
        'mean': [],
        'std': [],
        'min': [],
        'max': []
    }
    
    # Calculate statistics for each band
    for band_idx in range(tile_data.shape[0]):
        band = tile_data[band_idx]
        stats['mean'].append(np.mean(band))
        stats['std'].append(np.std(band))
        stats['min'].append(np.min(band))
        stats['max'].append(np.max(band))
    
    return stats

# Process all tiles
print("Processing tiles and calculating statistics...")
all_tile_stats = apply_processing_to_tiles(tiler, calculate_tile_statistics)

print(f"\nSuccessfully processed {len(all_tile_stats)} tiles")

# Display stats summary
print("\n" + "=" * 60)
print("TILE PROCESSING RESULTS")
print("=" * 60)
print(f"\nBand 1 (Red) Statistics across all tiles:")
band1_means = [s['mean'][0] for s in all_tile_stats if len(s['mean']) > 0]
print(f"  Mean intensity range: {min(band1_means):.1f} - {max(band1_means):.1f}")
print(f"  Average mean: {np.mean(band1_means):.1f}")

print(f"\nBand 2 (Green) Statistics:")
band2_means = [s['mean'][1] for s in all_tile_stats if len(s['mean']) > 1]
print(f"  Mean intensity range: {min(band2_means):.1f} - {max(band2_means):.1f}")
print(f"  Average mean: {np.mean(band2_means):.1f}")

print(f"\nBand 3 (Blue) Statistics:")
band3_means = [s['mean'][2] for s in all_tile_stats if len(s['mean']) > 2]
print(f"  Mean intensity range: {min(band3_means):.1f} - {max(band3_means):.1f}")
print(f"  Average mean: {np.mean(band3_means):.1f}")
print("=" * 60)

## 5. Process Individual Tiles

Apply image processing operations to each tile independently. This demonstrates how to calculate statistics per tile.

In [ ]:
# Visualize tile layout on the raster
fig, ax = plt.subplots(figsize=(14, 10))

with rasterio.open(raster_path) as src:
    show(src.read(1), ax=ax, cmap='gray', alpha=0.7)

# Draw tile boundaries
windows = tiler.get_tile_windows()
for i, window in enumerate(windows):
    # Convert window to bounds
    with rasterio.open(raster_path) as src:
        bounds = src.window_bounds(window)
    
    x_min, y_min, x_max, y_max = bounds
    
    # Draw rectangle
    from matplotlib.patches import Rectangle
    rect = Rectangle((x_min, y_min), x_max - x_min, y_max - y_min, 
                     linewidth=1, edgecolor='red', facecolor='none', alpha=0.5)
    ax.add_patch(rect)

ax.set_title(f'Raster Tiling Layout ({len(windows)} tiles)', fontsize=14)
ax.set_xlabel('X Coordinate')
ax.set_ylabel('Y Coordinate')
plt.tight_layout()
plt.savefig('Output/tile_layout.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"Tile layout visualization saved to Output/tile_layout.png")

In [ ]:
print("Loading all tiles...")
tiles, tile_metadata = tiler.read_all_tiles()

print(f"\nSuccessfully loaded {len(tiles)} tiles")
print(f"\nTile Information:")
print(f"  Number of tiles: {len(tiles)}")
print(f"  Shape of first tile: {tiles[0].shape}")
print(f"  Data type: {tiles[0].dtype}")

# Display information about first few tiles
print(f"\nFirst 3 tiles metadata:")
for i, meta in enumerate(tile_metadata[:3]):
    print(f"\n  Tile {i}:")
    print(f"    Position (pixel): row={meta['row_off']}, col={meta['col_off']}")
    print(f"    Size (pixels): {meta['width']}x{meta['height']}")
    print(f"    Bounds (CRS): {meta['bounds']}")

## 4. Split Raster into Tiles

Extract individual tiles from the raster with spatial reference information.

In [ ]:
# Define tiling parameters
TILE_SIZE = 512      # 512x512 pixel tiles
OVERLAP = 50         # 50 pixel overlap for seamless processing

print("=" * 60)
print("TILING PARAMETERS")
print("=" * 60)
print(f"Tile size: {TILE_SIZE}x{TILE_SIZE} pixels")
print(f"Overlap: {OVERLAP} pixels")
print(f"Step size: {TILE_SIZE - OVERLAP} pixels")

# Initialize tiler
tiler = RasterTiler(raster_path, tile_size=TILE_SIZE, overlap=OVERLAP)

# Get tiling statistics
stats = tiler.get_statistics()
print(f"\nTiling Statistics:")
print(f"  Number of tiles: {stats['num_tiles']}")
print(f"  Memory per tile: {stats['estimated_memory_per_tile_mb']:.1f} MB")
print(f"  Total tiles memory: {stats['estimated_memory_per_tile_mb'] * stats['num_tiles']:.1f} MB")
print(f"  Memory reduction vs full load: {(1 - stats['estimated_memory_per_tile_mb'] / raster_size_mb) * 100:.1f}%")
print("=" * 60)

## 3. Define Tile Parameters

Set tile size and overlap parameters for optimal processing.

In [ ]:
raster_path = 'Ortho/example.tif'

# Open and inspect raster
with rasterio.open(raster_path) as src:
    print("=" * 60)
    print("RASTER METADATA")
    print("=" * 60)
    print(f"File: {raster_path}")
    print(f"Dimensions: {src.width}x{src.height} pixels")
    print(f"Coordinate Reference System (CRS): {src.crs}")
    print(f"Number of bands: {src.count}")
    print(f"Data type: {src.dtypes[0]}")
    print(f"Transform: {src.transform}")
    print(f"Bounds: {src.bounds}")
    print(f"Resolution: {src.res[0]:.2f}m/pixel")
    print("=" * 60)
    
    # Calculate raster size
    raster_size_mb = (src.width * src.height * src.count * 
                      np.dtype(src.dtypes[0]).itemsize / 1024 / 1024)
    print(f"Estimated uncompressed size: {raster_size_mb:.1f} MB")

## 2. Load Raster Data

Load the orthophoto raster and examine its metadata.

In [2]:
import numpy as np
import rasterio
from rasterio.windows import Window
from rasterio.plot import show
import matplotlib.pyplot as plt
import geopandas as gpd
from pathlib import Path
import os
from raster_tiling import RasterTiler, apply_processing_to_tiles

print("Libraries imported successfully!")
print(f"Working directory: {os.getcwd()}")

Libraries imported successfully!
Working directory: /home/whoami/Documents/GitHub-Repo/Palm-Tree-Counting


## 1. Import Required Libraries

# Raster Tiling Workflow for Palm Tree Detection
## Splitting, Processing, and Merging Raster Data for Improved Accuracy

This notebook demonstrates how to split large rasters into manageable tiles, process them independently, and merge the results back together. This approach improves memory efficiency and processing accuracy for large geospatial datasets.

## Summary

This workflow demonstrates the complete raster tiling process:

1. **Load Raster Data**: Inspect large raster files and their properties
2. **Define Parameters**: Configure tile size and overlap for optimal processing
3. **Split into Tiles**: Create manageable chunks while maintaining spatial reference
4. **Process Tiles**: Apply operations independently to reduce memory usage
5. **Merge Results**: Combine tiles back into a single raster with overlap handling
6. **Validate**: Verify accuracy by comparing with the original raster

### Key Benefits:
- **Memory Efficiency**: Process one tile at a time instead of entire raster
- **Better Accuracy**: Overlaps help reduce edge effects in processing
- **Parallelization**: Tiles can be processed in parallel for speed improvements
- **Scalability**: Works with arbitrarily large rasters

### Next Steps:
You can integrate this tiling approach into your template matching workflow by:
- Processing template matching on each tile separately
- Merging detection results from all tiles
- Running BIRCH clustering on the merged detections

In [1]:
# Visualize error distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: MSE heatmap
im = axes[0].imshow(mse_per_pixel, cmap='hot')
axes[0].set_title('Pixel-wise MSE (Error Map)', fontsize=12)
plt.colorbar(im, ax=axes[0], label='MSE')

# Plot 2: MSE histogram
axes[1].hist(mse_per_pixel.flatten(), bins=100, edgecolor='black')
axes[1].set_xlabel('MSE Value')
axes[1].set_ylabel('Number of Pixels')
axes[1].set_title('Distribution of Pixel-wise Errors', fontsize=12)
axes[1].set_yscale('log')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('Output/error_analysis.png', dpi=100, bbox_inches='tight')
plt.show()

print("Error analysis visualization saved to Output/error_analysis.png")

NameError: name 'plt' is not defined

In [ ]:
# Visualize differences
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Show original bands
for band_idx in range(3):
    ax = axes[0, band_idx]
    band_data = original[band_idx].astype(np.uint8)
    ax.imshow(band_data, cmap='gray')
    ax.set_title(f'Original Band {band_idx + 1}')
    ax.axis('off')

# Show merged bands
for band_idx in range(3):
    ax = axes[1, band_idx]
    band_data = np.clip(merged_raster[band_idx], 0, 255).astype(np.uint8)
    ax.imshow(band_data, cmap='gray')
    ax.set_title(f'Merged Band {band_idx + 1}')
    ax.axis('off')

plt.suptitle('Comparison: Original vs Merged Bands', fontsize=14, y=0.98)
plt.tight_layout()
plt.savefig('Output/band_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

print("Band comparison visualization saved to Output/band_comparison.png")

In [ ]:
print("Validating merged raster against original...")
print("=" * 60)

# Load original raster for comparison
with rasterio.open(raster_path) as src:
    original = src.read().astype(np.float32)

# Compare statistics
print("\nStatistical Comparison (per band):")
print(f"{'Band':<6} {'Metric':<20} {'Original':<12} {'Merged':<12} {'Difference':<12}")
print("-" * 62)

for band_idx in range(num_bands):
    original_band = original[band_idx]
    merged_band = merged_raster[band_idx]
    
    orig_mean = np.mean(original_band)
    merged_mean = np.mean(merged_band)
    diff_mean = abs(orig_mean - merged_mean)
    
    print(f"Band {band_idx+1:<2} {'Mean':<20} {orig_mean:<12.2f} {merged_mean:<12.2f} {diff_mean:<12.2f}")
    
    orig_std = np.std(original_band)
    merged_std = np.std(merged_band)
    diff_std = abs(orig_std - merged_std)
    
    print(f"{'':6} {'Std Dev':<20} {orig_std:<12.2f} {merged_std:<12.2f} {diff_std:<12.2f}")

# Calculate overall RMSE
rmse = np.sqrt(np.mean((original - merged_raster) ** 2))
print(f"\nOverall RMSE: {rmse:.4f}")

# Calculate MSE per pixel to find problematic areas
mse_per_pixel = np.mean((original - merged_raster) ** 2, axis=0)

print(f"\nPixel-wise MSE Statistics:")
print(f"  Min MSE: {np.min(mse_per_pixel):.6f}")
print(f"  Max MSE: {np.max(mse_per_pixel):.6f}")
print(f"  Mean MSE: {np.mean(mse_per_pixel):.6f}")
print(f"  Median MSE: {np.median(mse_per_pixel):.6f}")

print("=" * 60)

## 7. Validate Results

Compare the merged raster with the original to verify accuracy and data integrity.

In [ ]:
def merge_tiles_simple(tiles, tile_metadata, raster_width, raster_height, num_bands):
    """
    Merge tiles back into single raster, averaging overlapping regions
    
    Args:
        tiles: List of tile arrays
        tile_metadata: List of tile metadata dicts
        raster_width: Width of original raster
        raster_height: Height of original raster
        num_bands: Number of bands
        
    Returns:
        Merged raster array and weight map
    """
    # Initialize output raster and weight map
    merged = np.zeros((num_bands, raster_height, raster_width), dtype=np.float32)
    weights = np.zeros((raster_height, raster_width), dtype=np.float32)
    
    # Place each tile into the merged raster
    for tile, meta in zip(tiles, tile_metadata):
        row_off = meta['row_off']
        col_off = meta['col_off']
        tile_height = meta['height']
        tile_width = meta['width']
        
        # Add tile data
        merged[:, row_off:row_off+tile_height, col_off:col_off+tile_width] += tile
        weights[row_off:row_off+tile_height, col_off:col_off+tile_width] += 1
    
    # Normalize by weights (handle overlapping regions)
    weights[weights == 0] = 1  # Avoid division by zero
    for band in range(num_bands):
        merged[band] = merged[band] / weights
    
    return merged, weights

print("Merging tiles back together...")

with rasterio.open(raster_path) as src:
    num_bands = src.count
    raster_width = src.width
    raster_height = src.height

merged_raster, weight_map = merge_tiles_simple(
    tiles, tile_metadata, raster_width, raster_height, num_bands
)

print(f"Merged raster shape: {merged_raster.shape}")
print(f"Merged raster data type: {merged_raster.dtype}")

# Visualize weight map (shows overlap regions)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# Plot 1: Weight map
im1 = ax1.imshow(weight_map, cmap='viridis')
ax1.set_title('Tile Weight Map (overlap regions)', fontsize=12)
plt.colorbar(im1, ax=ax1, label='Number of overlapping tiles')

# Plot 2: Merged raster (RGB composite)
rgb_merged = merged_raster[:3].astype(np.uint8).transpose(1, 2, 0)
rgb_merged = np.clip(rgb_merged, 0, 255)
ax2.imshow(rgb_merged)
ax2.set_title('Merged Raster (RGB)', fontsize=12)

plt.tight_layout()
plt.savefig('Output/merge_result.png', dpi=100, bbox_inches='tight')
plt.show()

print("Merge result visualization saved to Output/merge_result.png")

## 6. Merge Tiles Back Together

Combine processed tiles back into a single raster while handling overlaps correctly.

In [ ]:
def calculate_tile_statistics(tile_data, tile_meta):
    """
    Example processing function: calculate statistics for each tile
    
    Args:
        tile_data: numpy array of shape (bands, height, width)
        tile_meta: dict with tile metadata
        
    Returns:
        dict with statistics for this tile
    """
    stats = {
        'tile_position': (tile_meta['row_off'], tile_meta['col_off']),
        'band_count': tile_data.shape[0],
        'mean': [],
        'std': [],
        'min': [],
        'max': []
    }
    
    # Calculate statistics for each band
    for band_idx in range(tile_data.shape[0]):
        band = tile_data[band_idx]
        stats['mean'].append(np.mean(band))
        stats['std'].append(np.std(band))
        stats['min'].append(np.min(band))
        stats['max'].append(np.max(band))
    
    return stats

# Process all tiles
print("Processing tiles and calculating statistics...")
all_tile_stats = apply_processing_to_tiles(tiler, calculate_tile_statistics)

print(f"\nSuccessfully processed {len(all_tile_stats)} tiles")

# Display stats summary
print("\n" + "=" * 60)
print("TILE PROCESSING RESULTS")
print("=" * 60)
print(f"\nBand 1 (Red) Statistics across all tiles:")
band1_means = [s['mean'][0] for s in all_tile_stats if len(s['mean']) > 0]
print(f"  Mean intensity range: {min(band1_means):.1f} - {max(band1_means):.1f}")
print(f"  Average mean: {np.mean(band1_means):.1f}")

print(f"\nBand 2 (Green) Statistics:")
band2_means = [s['mean'][1] for s in all_tile_stats if len(s['mean']) > 1]
print(f"  Mean intensity range: {min(band2_means):.1f} - {max(band2_means):.1f}")
print(f"  Average mean: {np.mean(band2_means):.1f}")

print(f"\nBand 3 (Blue) Statistics:")
band3_means = [s['mean'][2] for s in all_tile_stats if len(s['mean']) > 2]
print(f"  Mean intensity range: {min(band3_means):.1f} - {max(band3_means):.1f}")
print(f"  Average mean: {np.mean(band3_means):.1f}")
print("=" * 60)

## 5. Process Individual Tiles

Apply image processing operations to each tile independently. This demonstrates how to calculate statistics per tile.

In [ ]:
# Visualize tile layout on the raster
fig, ax = plt.subplots(figsize=(14, 10))

with rasterio.open(raster_path) as src:
    show(src.read(1), ax=ax, cmap='gray', alpha=0.7)

# Draw tile boundaries
windows = tiler.get_tile_windows()
for i, window in enumerate(windows):
    # Convert window to bounds
    with rasterio.open(raster_path) as src:
        bounds = src.window_bounds(window)
    
    x_min, y_min, x_max, y_max = bounds
    
    # Draw rectangle
    from matplotlib.patches import Rectangle
    rect = Rectangle((x_min, y_min), x_max - x_min, y_max - y_min, 
                     linewidth=1, edgecolor='red', facecolor='none', alpha=0.5)
    ax.add_patch(rect)

ax.set_title(f'Raster Tiling Layout ({len(windows)} tiles)', fontsize=14)
ax.set_xlabel('X Coordinate')
ax.set_ylabel('Y Coordinate')
plt.tight_layout()
plt.savefig('Output/tile_layout.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"Tile layout visualization saved to Output/tile_layout.png")

In [ ]:
print("Loading all tiles...")
tiles, tile_metadata = tiler.read_all_tiles()

print(f"\nSuccessfully loaded {len(tiles)} tiles")
print(f"\nTile Information:")
print(f"  Number of tiles: {len(tiles)}")
print(f"  Shape of first tile: {tiles[0].shape}")
print(f"  Data type: {tiles[0].dtype}")

# Display information about first few tiles
print(f"\nFirst 3 tiles metadata:")
for i, meta in enumerate(tile_metadata[:3]):
    print(f"\n  Tile {i}:")
    print(f"    Position (pixel): row={meta['row_off']}, col={meta['col_off']}")
    print(f"    Size (pixels): {meta['width']}x{meta['height']}")
    print(f"    Bounds (CRS): {meta['bounds']}")

## 4. Split Raster into Tiles

Extract individual tiles from the raster with spatial reference information.

In [ ]:
# Define tiling parameters
TILE_SIZE = 512      # 512x512 pixel tiles
OVERLAP = 50         # 50 pixel overlap for seamless processing

print("=" * 60)
print("TILING PARAMETERS")
print("=" * 60)
print(f"Tile size: {TILE_SIZE}x{TILE_SIZE} pixels")
print(f"Overlap: {OVERLAP} pixels")
print(f"Step size: {TILE_SIZE - OVERLAP} pixels")

# Initialize tiler
tiler = RasterTiler(raster_path, tile_size=TILE_SIZE, overlap=OVERLAP)

# Get tiling statistics
stats = tiler.get_statistics()
print(f"\nTiling Statistics:")
print(f"  Number of tiles: {stats['num_tiles']}")
print(f"  Memory per tile: {stats['estimated_memory_per_tile_mb']:.1f} MB")
print(f"  Total tiles memory: {stats['estimated_memory_per_tile_mb'] * stats['num_tiles']:.1f} MB")
print(f"  Memory reduction vs full load: {(1 - stats['estimated_memory_per_tile_mb'] / raster_size_mb) * 100:.1f}%")
print("=" * 60)

## 3. Define Tile Parameters

Set tile size and overlap parameters for optimal processing.

In [ ]:
raster_path = 'Ortho/example.tif'

# Open and inspect raster
with rasterio.open(raster_path) as src:
    print("=" * 60)
    print("RASTER METADATA")
    print("=" * 60)
    print(f"File: {raster_path}")
    print(f"Dimensions: {src.width}x{src.height} pixels")
    print(f"Coordinate Reference System (CRS): {src.crs}")
    print(f"Number of bands: {src.count}")
    print(f"Data type: {src.dtypes[0]}")
    print(f"Transform: {src.transform}")
    print(f"Bounds: {src.bounds}")
    print(f"Resolution: {src.res[0]:.2f}m/pixel")
    print("=" * 60)
    
    # Calculate raster size
    raster_size_mb = (src.width * src.height * src.count * 
                      np.dtype(src.dtypes[0]).itemsize / 1024 / 1024)
    print(f"Estimated uncompressed size: {raster_size_mb:.1f} MB")

## 2. Load Raster Data

Load the orthophoto raster and examine its metadata.

## 1. Import Required Libraries

In [ ]:
import numpy as np
import rasterio
from rasterio.windows import Window
from rasterio.plot import show
import matplotlib.pyplot as plt
import geopandas as gpd
from pathlib import Path
import os
from raster_tiling import RasterTiler, apply_processing_to_tiles

print("Libraries imported successfully!")
print(f"Working directory: {os.getcwd()}")

# Raster Tiling Workflow for Palm Tree Detection
## Splitting, Processing, and Merging Raster Data for Improved Accuracy

This notebook demonstrates how to split large rasters into manageable tiles, process them independently, and merge the results back together. This approach improves memory efficiency and processing accuracy for large geospatial datasets.